In [1]:
# ==============================================================================
# Script: extract_notame.py (Python / rpy2 Implementation)
# Description: Python equivalent of the finalized R script. Extracts intensity 
#              and metadata from notame S4 objects, utilizes Pandas for strict 
#              column filtering and standardization, and exports pipeline-ready 
#              CSVs.
# 
# Development & Tested Environment:
#   * Note: The following explicitly lists the module versions used during 
#     development and testing. It does NOT imply exhaustive testing across 
#     newer (>=) or older versions.
#
#   - Python     : == 3.13.13
#   - pandas     : == 3.0.3
#   - loguru     : == 0.7.3
#   - rpy2       : == 3.6.7 (Strictly requires modern localconverter paradigm)
#   - R Base     : == 4.5.3
#   - R Packages : SummarizedExperiment (Bioconductor), Biobase
# ==============================================================================


import os
import re
import sys
import pandas as pd
import rpy2.robjects as robjects
from rpy2.robjects import default_converter
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr
from loguru import logger
from pprint import pprint

# Configure loguru for a clean, concise terminal output
logger.remove()
logger.add(
    sys.stderr, 
    format=(
        "<green>{time:HH:mm:ss}</green> | "
        "<level>{level: <8}</level> | "
        "<level>{message}</level>")
)

def extract_and_standardize_rda(rda_name: str) -> tuple:
    """
    Extracts and standardizes intensity and metadata from a notame S4 object.
    
    Args:
        rda_name (str): Absolute or relative path to the .rda file.
        
    Returns:
        tuple: (intensity_df, meta_df) as standardized Pandas DataFrames,
        or (None, None) if extraction fails.
    """
    if not os.path.exists(rda_name):
        logger.error(f"File not found: {rda_name}")
        return None, None
    
    # Load required R packages for S4 object handling
    try:
        importr('SummarizedExperiment')
        importr('Biobase')
        r_base = importr('base') # Import base R to use as.data.frame
    except Exception as e:
        logger.error(f"Required R packages missing: {e}")
        return None, None

    logger.info(f"Loading and parsing {os.path.basename(rda_name)}...")
    
    # Load RDA into the R environment
    loaded_objects = robjects.r['load'](rda_name)
    if len(loaded_objects) == 0:
        logger.error("No valid R objects found inside the RDA file.")
        return None, None
        
    # Retrieve the first loaded object dynamically
    target_data = robjects.r[loaded_objects[0]]
    
    # 1. Extract raw matrices based on S4 Class type
    is_se = robjects.r['inherits'](target_data, "SummarizedExperiment")[0]
    if is_se:
        raw_r_intensity = robjects.r['assay'](target_data)
        raw_r_meta = robjects.r['colData'](target_data)
    else:
        raw_r_intensity = robjects.r['exprs'](target_data)
        raw_r_meta = robjects.r['pData'](target_data)

    logger.info(
        "Extracting and refactoring specific metadata columns via Pandas...")

    # ==========================================================================
    # Phase A: Metadata Refactoring and Strict Extraction
    # ==========================================================================
    
    # Coerce the Bioconductor S4 DataFrame into a standard R data.frame
    r_meta_df = r_base.as_data_frame(raw_r_meta)
    
    # Extract row names explicitly from R before conversion
    meta_rownames = list(robjects.r['rownames'](r_meta_df))

    # Safely convert standard R data.frame to Pandas DataFrame
    with localconverter(default_converter + pandas2ri.converter):
        meta_df = robjects.conversion.rpy2py(r_meta_df)
    
    # Assign the extracted rownames to the Pandas DataFrame index
    meta_df.index = meta_rownames
    
    # Convert underscores to spaces in Sample IDs
    clean_sample_names = [str(name).replace("_", " ") for name in meta_df.index]
    
    # Map and rename existing columns based on keywords
    new_columns = {}
    for col in meta_df.columns:
        lower_col = str(col).lower()
        if "batch" in lower_col:
            new_columns[col] = "Batch"
        elif "order" in lower_col:
            new_columns[col] = "Inject Order"
        elif "group" in lower_col:
            new_columns[col] = "Bio Group"
        elif "qc" in lower_col or lower_col == "class" or "type" in lower_col:
            new_columns[col] = "Sample Type"
        else:
            new_columns[col] = col
            
    meta_df.rename(columns=new_columns, inplace=True)
    
    # Explicitly bind the "Sample Name" column at the first position
    meta_df.insert(0, 'Sample Name', clean_sample_names)
    
    # Define the strict whitelist and keep only existing columns
    target_columns = [
        "Sample Name", "Inject Order", "Bio Group", "Sample Type", "Batch"]
    available_columns = [
        col for col in target_columns if col in meta_df.columns]
    meta_df = meta_df[available_columns].copy()
    
    # Standardize the values inside the "Sample Type" column
    if "Sample Type" in meta_df.columns:
        def std_type(val):
            if pd.isna(val): return val
            val_str = str(val).lower()
            if val_str in ["subject", "true sample", "sample"]:
                return "Sample"
            if val_str == "qc": return "QC"
            if val_str == "blank": return "Blank"
            return val
        meta_df["Sample Type"] = meta_df["Sample Type"].apply(std_type)

    # Standardize Batch values (e.g., "Batch 1" -> "Batch-1")
    if "Batch" in meta_df.columns:
        def std_batch(val):
            if pd.isna(val): return val
            num_str = re.sub(r'(?i)batch[-_\s]?', '', str(val))
            return f"Batch-{num_str}"
        meta_df["Batch"] = meta_df["Batch"].apply(std_batch)

    # Replace "QC" with "Invalid" in the "Bio Group" column
    if "Bio Group" in meta_df.columns:
        def std_group(val):
            if pd.isna(val): return val
            if str(val).lower() == "qc": return "Invalid"
            return val
        meta_df["Bio Group"] = meta_df["Bio Group"].apply(std_group)

    # Clear the index to mimic tibble behavior and ensure clean CSV export
    meta_df.reset_index(drop=True, inplace=True)

    # ==========================================================================
    # Phase B: Intensity Matrix Standardization
    # ==========================================================================
    
    # Extract matrix dimensions and names explicitly from R
    intensity_rownames = list(robjects.r['rownames'](raw_r_intensity))
    intensity_colnames = list(robjects.r['colnames'](raw_r_intensity))

    # Convert R matrix to a NumPy array safely
    with localconverter(default_converter + pandas2ri.converter):
        r_intensity_array = robjects.conversion.rpy2py(raw_r_intensity)
    
    # Construct the Pandas DataFrame explicitly using the NumPy array and names
    intensity_df = pd.DataFrame(
        data=r_intensity_array,
        index=intensity_rownames,
        columns=intensity_colnames
    )
    
    # Ensure column names (Sample IDs) match the cleaned version with spaces
    intensity_df.columns = clean_sample_names
    
    # Convert underscores in Metabolite/Feature IDs to spaces
    clean_metabolites = [
        str(name).replace("_", " ") for name in intensity_df.index]
    
    # Explicitly add the "Metabolite" column at the first position
    intensity_df.insert(0, 'Metabolite', clean_metabolites)
    
    # Clear the index to mimic tibble behavior
    intensity_df.reset_index(drop=True, inplace=True)

    # ==========================================================================
    # Phase C: Final Return
    # ==========================================================================
    logger.success(
        "Metadata columns strictly filtered and converted to DataFrames.")
    return intensity_df, meta_df


def process_and_export_rda(
    rda_name: str, input_dir: str, output_dir: str) -> bool:
    """
    Wrapper function to process a single RDA file and export the standardized
    DataFrames to CSV files.
    """
    rda_file_path = os.path.join(input_dir, f"{rda_name}.rda")
    
    if not os.path.exists(rda_file_path):
        logger.warning(f"File not found, skipping: {rda_file_path}")
        return False
        
    logger.info(f"Processing dataset: {rda_name}")
    
    # Call the core extraction and standardization function 
    res = extract_and_standardize_rda(rda_file_path)
    
    # Check if the first element is None instead of comparing the whole tuple
    if res[0] is None:
        return False
        
    intensity_df, meta_df = res
    
    # Construct dynamic output filenames using os.path.normpath for safety
    out_intensity_file = os.path.normpath(
        os.path.join(output_dir, f"project_intensity.csv"))
    out_meta_file = os.path.normpath(
        os.path.join(output_dir, f"project_meta.csv"))
    
    # Export to CSV without the pandas index
    intensity_df.to_csv(out_intensity_file, index=False)
    logger.success(
        f"Exported '{rda_name}' intensity file to: '{out_intensity_file}'")
    
    meta_df.to_csv(out_meta_file, index=False)
    logger.success(f"Exported '{rda_name}' meta file to: '{out_meta_file}'")
    
    return intensity_df, meta_df


# ==============================================================================
# Batch Execution Pipeline (The Main Script)
# ==============================================================================
# if __name__ == "__main__":
# 1. Define base directories (Jupyter Notebook Compatible)
notebook_dir = os.path.abspath(os.getcwd())

input_dir = os.path.normpath(
    os.path.join(notebook_dir, "..", "..", "data", "raw", "notame"))
output_dir = os.path.normpath(
    os.path.join(notebook_dir, "..", "..", "data", "processed", "notame"))

# Safely create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# 2. Define the target dataset
ds_name = "toy_notame_set"

# 3. Execute the batch processing
intensity_df, metadata_df = process_and_export_rda(
    rda_name=ds_name,
    input_dir=input_dir,
    output_dir=output_dir
)

logger.success("\n[INFO] DATASETS PROCESSED AND EXPORTED SUCCESSFULLY! 🎉")

16:21:00 | INFO     | Processing dataset: toy_notame_set
16:21:03 | INFO     | Loading and parsing toy_notame_set.rda...
16:21:03 | INFO     | Extracting and refactoring specific metadata columns via Pandas...
16:21:03 | SUCCESS  | Metadata columns strictly filtered and converted to DataFrames.
16:21:03 | SUCCESS  | Exported 'toy_notame_set' intensity file to: 'c:\Users\Complex\Desktop\Personal_repositories\pi-metaboqc-casestudy\data\processed\notame\project_intensity.csv'
16:21:03 | SUCCESS  | Exported 'toy_notame_set' meta file to: 'c:\Users\Complex\Desktop\Personal_repositories\pi-metaboqc-casestudy\data\processed\notame\project_meta.csv'
16:21:03 | SUCCESS  | 
[INFO] DATASETS PROCESSED AND EXPORTED SUCCESSFULLY! 🎉


In [2]:
pprint(metadata_df.head(5))

  Sample Name  Inject Order Bio Group Sample Type    Batch
0      Demo 1             1   Invalid          QC  Batch-1
1      Demo 2             2         B      Sample  Batch-1
2      Demo 3             3         B      Sample  Batch-1
3      Demo 4             4         B      Sample  Batch-1
4      Demo 5             5         A      Sample  Batch-1


In [3]:
pprint(intensity_df.shape)
pprint(intensity_df.head(5))

(80, 51)
                  Metabolite        Demo 1        Demo 2        Demo 3  \
0  HILIC neg 259 9623a4 4322  30873.825778  30193.284725      0.000000   
1  HILIC neg 108 1065a2 6121  11517.655210   4820.770234  14747.642390   
2    HILIC neg 158 23a1 4128  20690.591985  18929.531778  16823.094855   
3  HILIC neg 251 0056a0 6161  14816.347685  37198.204938  26748.505390   
4     HILIC neg 401 52a4 211  22093.182271   9927.692412  11526.047544   

         Demo 4        Demo 5        Demo 6        Demo 7        Demo 8  \
0  48290.002578  22991.497754  34749.475841  37515.310643  39178.998716   
1  13732.419727   4043.557353  16292.051433      0.000000  12656.521817   
2  26141.084001  25379.050434      0.000000  14208.226394  12058.220001   
3  20747.948990  29489.259553  20958.135153  15830.217957  12710.173671   
4  23915.147726  19830.849925  19028.935044  18783.514257  15180.038952   

         Demo 9  ...       Demo 41       Demo 42       Demo 43       Demo 44  \
0  38574.301109